# 03 --- Tool Scoping: 4-5 Tools Per Agent

**CCA Pattern**: Each agent gets 4-5 focused tools. The super agent anti-pattern
(18 or more tools on one agent; ours has 20) causes tool selection degradation.

The exam's official guidance is specific: **keep 4-5 tools per agent**.
This is the exam-correct answer.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
import json

from research_agents.tools.definitions import ALL_TOOL_SETS
from research_agents.tools.handlers import dispatch
from research_agents.anti_patterns.super_agent import (
    SUPER_AGENT_TOOLS, get_super_agent_tool_count, super_agent_dispatch,
)
from research_agents.agent.agent_loop import run_agent_loop
from research_agents.agent.subagents import SUBAGENT_CONFIGS
from research_agents.services.container import make_default_services
from research_agents.testing import scripted_client, text_turn, tool_turn

services = make_default_services()

## How Tool Definitions Work

Tool definitions live in `tools/definitions.py`. Each agent type has a constant
list of tool dicts, for example `WEB_RESEARCHER_TOOLS`.

Each tool dict follows the Claude API format:

```python
{
    "name": "search_web",
    "description": (
        "Search the public web for information matching a query. "
        "Returns URLs, titles, and snippets. "
        "Does NOT fetch full page content (use fetch_page for that). "
        "Does NOT search internal documents or databases."
    ),
    "input_schema": { ... }
}
```

### Negative-Bound Descriptions

Every tool description includes **"Does NOT..."** clauses. This is the CCA
negative-bound pattern -- it tells Claude what a tool *cannot* do, preventing
misrouting. Without these, Claude might call `search_web` when it needs
`parse_document`, because both involve finding information.

### The Dispatch Registry

Tool handlers are dispatched via a nested dict in `tools/handlers.py`:

```python
DISPATCH: dict[str, dict[str, Handler]] = {
    "web_researcher": {
        "search_web": handle_search_web,
        "fetch_page": handle_fetch_page,
        ...
    },
    ...
}
```

The key insight: dispatch is keyed by agent type *then* tool name. A tool is
only reachable through the agent that owns it, and an out-of-scope call gets a
structured error, not a handler. Tools are isolated per agent at the dispatch
level, which is enforcement in code rather than a request in a prompt.

## Anti-Pattern: Super Agent with 18+ Tools

In [ ]:
# The super agent has ALL tools combined
print(f'Super agent tool count: {get_super_agent_tool_count()}')
print()
print('All tools on one agent:')
for i, tool in enumerate(SUPER_AGENT_TOOLS, 1):
    print(f'  {i:2d}. {tool["name"]:<25s} {tool["description"][:60]}...')

### Why 18+ Tools Fails

When an agent has 18 or more tools (20 here):
- A significant portion of attention goes to evaluating tool descriptions
instead of the actual task
- Similar tools create ambiguity (e.g., `search_web` vs `cross_reference` --
both find information)
- **Better descriptions don't fix the structural problem** -- this is a
key exam distractor

Those are claims about model behaviour, and this notebook cannot measure
them without a live model. What it *can* show is the structural half: with
one giant tool list there is nothing in the code that stops a wrong choice
from executing. The replay below does exactly that.

## The Show Truck vs. Work Truck Analogy

A useful mental model for the exam: imagine a mechanic arriving at a job site.

- **The show truck** is magnificent. It carries *every tool the mechanic owns* --
drawer after drawer of sockets, a dozen types of wrench, diagnostic computers, a
lift and a press. Impressive. Also: every time the mechanic needs a 10mm
socket, they search through every drawer. Every minute spent searching is a
minute not spent working. The show truck looks capable but is *operationally*
slow because selection cost dominates work cost.

- **The work truck** carries exactly the 4-5 tools for today's job. The
mechanic reaches without looking. No selection overhead. If tomorrow's job
needs different tools, they swap the loadout -- they don't bolt on more
drawers.

Every specialized agent in this project is a work truck. The `super_agent`
anti-pattern is a show truck. The CCA exam's correct answer is always
"give the mechanic the right work truck" -- not "improve the labels on
the show truck's drawers."

Concretely in this codebase: `WEB_RESEARCHER_TOOLS` is the web researcher's
work truck. `DOCUMENT_ANALYZER_TOOLS` is the document analyzer's. Each set
has exactly 4 tools. When the coordinator dispatches a task, it hands the
*right* work truck to the *right* mechanic. That handoff is structural,
not suggestive -- the dispatch registry in `tools/handlers.py` is nested by
agent type and then tool name, so a tool that is not on this agent's truck
cannot be reached from it.

## Correct Pattern: Focused Tool Sets

In [ ]:
# Each subagent has 4 focused tools. The coordinator has its own set of 4 too.
print('Focused tool sets:')
for agent_type, tools in ALL_TOOL_SETS.items():
    tool_names = [t['name'] for t in tools]
    role = 'subagent' if agent_type in SUBAGENT_CONFIGS else 'hub'
    print(f'  {agent_type:<20s} ({len(tools)} tools, {role}): {", ".join(tool_names)}')

### What scoping enforces: replay one transcript through both routers

The scripted transcript below is a web researcher that decides to call
`query_database` -- a data extractor's tool. The same transcript runs through
the real loop twice. The only difference is the router: the super agent's flat
map, or the scoped `dispatch`.

In [ ]:
transcript = [
    tool_turn('query_database', {'table': 'remote_work_stats'}),
    text_turn('Done.'),
]
web_config = SUBAGENT_CONFIGS['web_researcher']

def replay(dispatch_fn) -> dict:
    client = scripted_client(list(transcript))
    result = run_agent_loop(client, services, 'Find remote work statistics',
                            web_config.system_prompt, web_config.tools,
                            'web_researcher', dispatch_fn=dispatch_fn)
    return json.loads(result.tool_results[0]['result'])

super_result = replay(super_agent_dispatch)
scoped_result = replay(dispatch)

print(f"Super agent:  {super_result['status']} -- "
      f"{len(super_result['data']['rows'])} rows returned from the database")
print(f"Scoped agent: {scoped_result['status']} -- {scoped_result['message']}")

In [ ]:
from helpers import compare_results

compare_results(
    {'max_tools_per_agent': get_super_agent_tool_count(),
     'subagents': 1,
     'out_of_scope_call_blocked': super_result['status'] == 'error'},
    {'max_tools_per_agent': max(len(c.tools) for c in SUBAGENT_CONFIGS.values()),
     'subagents': len(SUBAGENT_CONFIGS),
     'out_of_scope_call_blocked': scoped_result['status'] == 'error'},
)

## CCA Exam Tip

> The super agent question is one of the most reliable on the CCA exam.
> Answer choices will include:
> - 'improve tool descriptions' -- WRONG
> - 'add a tool-selection preprocessing step' -- WRONG
> - **'decompose into specialized subagents with 4-5 tools each' -- CORRECT**
> The improvement is architectural, not descriptive.

*Why architectural?* The root cause is **attention fragmentation** --
an 18-tool menu dilutes the model's focus regardless of how well each
tool is described. Sharper descriptions are a band-aid; fewer, scoped
tools fix the mechanism.

*What the code proves vs. what the exam claims:* the replay above shows
scoping is enforced at dispatch (a wrong pick is refused), which is the part
you can test without a model. The claim that fewer tools also make the
*right* pick more likely is the exam's, and it is why the fix is structural.